# Drug Response Random Forest

This notebook mirrors the structure of `4_Model_implementatin.ipynb` but trains a plain scikit-learn multi-output `RandomForestRegressor` instead of a PyTorch Lightning model. It reuses the same data loading, split assignment, and preprocessor fitting so results can be compared fairly against the Ridge/MLP runs in notebook 4.

The random forest takes **L1000-only** inputs (977 baseline genes + 2048 Morgan fingerprint bits + 1 scaled dose = 3026 features) and predicts the 977-dim delta expression directly (no standardization).


In [ ]:
from pathlib import Path

SPLIT_MODE = "drug_blind"  # one of {"drug_blind", "tumor_blind", "mixed"}
SPLIT_FRACTIONS = {"train": 0.8, "val": 0.1, "test": 0.1}
RANDOM_SEED = 42
NUM_WORKERS = 0

MODEL_NAME = "random_forest"
TARGET_MODE = "delta"
PREPROCESS_BATCH_SIZE = 1024

RF_N_ESTIMATORS = 100
RF_MAX_DEPTH = 20
RF_MIN_SAMPLES_LEAF = 5
RF_MAX_FEATURES = "sqrt"
RF_N_JOBS = -1
RF_OOB_SCORE = True
TRAIN_ROW_SUBSAMPLE_FRACTION = None  # set to a float in (0, 1] to subsample training rows for smoke runs

PLOT_TEST_SAMPLE_COUNT = 50
PLOT_TEST_SAMPLE_STRATEGY = "seeded_random"
PLOT_RANDOM_SEED = RANDOM_SEED
EMBEDDING_METHOD = "pca"
N_EMBEDDING_COMPONENTS = 2

TENSOR_ARTIFACTS_DIR = Path("data/Tahoe100M_tensor_artifacts_L1000")
SKLEARN_OUTPUT_DIR = Path("artifacts/sklearn/random_forest")

print(
    {
        "SPLIT_MODE": SPLIT_MODE,
        "SPLIT_FRACTIONS": SPLIT_FRACTIONS,
        "RANDOM_SEED": RANDOM_SEED,
        "NUM_WORKERS": NUM_WORKERS,
        "MODEL_NAME": MODEL_NAME,
        "TARGET_MODE": TARGET_MODE,
        "PREPROCESS_BATCH_SIZE": PREPROCESS_BATCH_SIZE,
        "RF_N_ESTIMATORS": RF_N_ESTIMATORS,
        "RF_MAX_DEPTH": RF_MAX_DEPTH,
        "RF_MIN_SAMPLES_LEAF": RF_MIN_SAMPLES_LEAF,
        "RF_MAX_FEATURES": RF_MAX_FEATURES,
        "RF_N_JOBS": RF_N_JOBS,
        "RF_OOB_SCORE": RF_OOB_SCORE,
        "TRAIN_ROW_SUBSAMPLE_FRACTION": TRAIN_ROW_SUBSAMPLE_FRACTION,
        "PLOT_TEST_SAMPLE_COUNT": PLOT_TEST_SAMPLE_COUNT,
        "PLOT_TEST_SAMPLE_STRATEGY": PLOT_TEST_SAMPLE_STRATEGY,
        "PLOT_RANDOM_SEED": PLOT_RANDOM_SEED,
        "EMBEDDING_METHOD": EMBEDDING_METHOD,
        "N_EMBEDDING_COMPONENTS": N_EMBEDDING_COMPONENTS,
        "tumor_blind_behavior": "cell_line_blind",
        "model_family": "sklearn_random_forest",
    }
)


## Imports and Helpers


In [ ]:
import json
import sys
import time
from pathlib import Path

import joblib
import lightning as L
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor

for candidate_root in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate_root / "pyproject.toml").exists() and (candidate_root / "Machine_Learning" / "ml_pipeline").exists():
        candidate_root_str = str(candidate_root)
        if candidate_root_str not in sys.path:
            sys.path.insert(0, candidate_root_str)
        break
else:
    raise ModuleNotFoundError(
        "Could not resolve the project root needed to import Machine_Learning.ml_pipeline."
    )

from Machine_Learning.ml_pipeline.data import TrainingPreprocessor
from Machine_Learning.ml_pipeline.utils import (
    SPLIT_NAMES,
    assign_group_blind_splits,
    assign_mixed_split,
    build_overlap_diagnostics,
    build_project_path,
    build_rf_training_arrays,
    build_sklearn_prediction_pair_embedding,
    build_split_summary,
    evaluate_sklearn_predictions,
    load_cached_cell_line_metadata,
    resolve_project_path,
    resolve_target_gene_indices,
    sample_prediction_details,
    validate_plot_config,
    validate_split_assignments,
    validate_split_config,
)

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")


## Load Tensors and Supporting Metadata


In [ ]:
validate_split_config(SPLIT_MODE, SPLIT_FRACTIONS)
validate_plot_config(
    PLOT_TEST_SAMPLE_COUNT,
    PLOT_TEST_SAMPLE_STRATEGY,
    EMBEDDING_METHOD,
    N_EMBEDDING_COMPONENTS,
)
if MODEL_NAME != "random_forest":
    raise ValueError(f"This notebook only supports MODEL_NAME='random_forest'; got {MODEL_NAME}")
if TARGET_MODE != "delta":
    raise ValueError(f"This notebook only supports TARGET_MODE='delta'; got {TARGET_MODE}")
if TRAIN_ROW_SUBSAMPLE_FRACTION is not None and not (0.0 < float(TRAIN_ROW_SUBSAMPLE_FRACTION) <= 1.0):
    raise ValueError("TRAIN_ROW_SUBSAMPLE_FRACTION must be None or a float in (0, 1].")

tensor_artifacts_dir = resolve_project_path(TENSOR_ARTIFACTS_DIR)
dmso_bundle = torch.load(tensor_artifacts_dir / "dmso_baselines.pt", map_location="cpu")
treatment_bundle = torch.load(tensor_artifacts_dir / "treatment_expressions.pt", map_location="cpu")
fingerprint_bundle = torch.load(tensor_artifacts_dir / "morgan_fingerprints.pt", map_location="cpu")

input_gene_ids = [str(gene_id) for gene_id in dmso_bundle["gene_ids"]]
target_gene_ids = [str(gene_id) for gene_id in treatment_bundle["gene_ids"]]
target_gene_indices = resolve_target_gene_indices(input_gene_ids, target_gene_ids)

treatment_feature_space = str(treatment_bundle.get("feature_space", "unknown"))
available_landmark_gene_ids = [
    str(gene_id) for gene_id in treatment_bundle.get("available_landmark_gene_ids", target_gene_ids)
]
if available_landmark_gene_ids and target_gene_ids != available_landmark_gene_ids:
    raise ValueError("Treatment target gene_ids do not match available_landmark_gene_ids metadata.")
if len(target_gene_indices) != len(target_gene_ids):
    raise ValueError("Resolved target gene indices do not match the saved treatment target width.")

cell_line_metadata_df, cell_line_metadata_arrow_path = load_cached_cell_line_metadata()
tensor_cell_lines = set(dmso_bundle["cell_lines"])
matched_cell_line_metadata_df = cell_line_metadata_df.loc[
    cell_line_metadata_df["cell_line"].isin(tensor_cell_lines)
].copy()
missing_cell_line_metadata = sorted(tensor_cell_lines - set(matched_cell_line_metadata_df["cell_line"]))
if missing_cell_line_metadata:
    raise ValueError(f"Missing cell-line metadata for: {missing_cell_line_metadata}")
if matched_cell_line_metadata_df["cell_line"].duplicated().any():
    raise ValueError("cell_line metadata must be unique after deduplication.")

examples_df = pd.DataFrame(
    {
        "condition_key": treatment_bundle["condition_keys"],
        "cell_line": treatment_bundle["cell_lines"],
        "file_name": treatment_bundle["file_names"],
        "drug": treatment_bundle["drug_names"],
        "concentration": treatment_bundle["concentrations"].cpu().numpy().astype(np.float32),
        "concentration_unit": treatment_bundle["concentration_units"],
        "target_index": np.arange(len(treatment_bundle["condition_keys"]), dtype=np.int64),
    }
)
examples_df["baseline_index"] = examples_df["cell_line"].map(dmso_bundle["cell_line_to_index"])
examples_df["fingerprint_index"] = examples_df["drug"].map(fingerprint_bundle["drug_to_index"])

if examples_df["condition_key"].duplicated().any():
    raise ValueError("condition_key values must be unique in the treatment bundle.")
if examples_df["baseline_index"].isna().any():
    raise ValueError("Some treatment rows do not resolve to a DMSO baseline index.")
if examples_df["fingerprint_index"].isna().any():
    raise ValueError("Some treatment rows do not resolve to a Morgan fingerprint index.")

examples_df = examples_df.merge(
    matched_cell_line_metadata_df,
    on="cell_line",
    how="left",
    validate="many_to_one",
)
if examples_df[["cell_name", "organ"]].isna().any().any():
    raise ValueError("Some treatment rows do not resolve to cell-line metadata.")

examples_df[["baseline_index", "fingerprint_index", "target_index"]] = examples_df[[
    "baseline_index",
    "fingerprint_index",
    "target_index",
]].astype(int)

tensor_summary_df = pd.DataFrame(
    [
        {
            "bundle": "dmso_baselines",
            "rows": int(dmso_bundle["expressions"].shape[0]),
            "cols": int(dmso_bundle["expressions"].shape[1]),
            "feature_space": "full_protein_coding",
        },
        {
            "bundle": "treatment_expressions",
            "rows": int(treatment_bundle["expressions"].shape[0]),
            "cols": int(treatment_bundle["expressions"].shape[1]),
            "feature_space": treatment_feature_space,
        },
        {
            "bundle": "morgan_fingerprints",
            "rows": int(fingerprint_bundle["fingerprints"].shape[0]),
            "cols": int(fingerprint_bundle["fingerprints"].shape[1]),
            "feature_space": "morgan_radius2_bits2048",
        },
    ]
)
metadata_summary_df = pd.DataFrame(
    [
        {
            "tensor_artifacts_dir": str(tensor_artifacts_dir),
            "cell_line_metadata_arrow_path": str(cell_line_metadata_arrow_path),
            "n_examples": int(len(examples_df)),
            "n_unique_drugs": int(examples_df["drug"].nunique()),
            "n_unique_cell_lines": int(examples_df["cell_line"].nunique()),
            "n_unique_organs": int(examples_df["organ"].nunique()),
        }
    ]
)
print(
    f"Loaded {len(examples_df)} treatment examples with {len(input_gene_ids)} input genes and {len(target_gene_ids)} target genes in feature space {treatment_feature_space}."
)
display(tensor_summary_df)
display(metadata_summary_df)
display(examples_df.head())


## Split Examples

Same split logic as notebook 4, but no DataLoader construction since the Random Forest is fit on flat numpy arrays.


In [ ]:
if SPLIT_MODE == "drug_blind":
    split_mode_note = "drug_blind: entire drugs are held out from train."
    split_unit_column = "drug"
    split_assignments = assign_group_blind_splits(
        examples_df,
        group_col="drug",
        split_fractions=SPLIT_FRACTIONS,
        seed=RANDOM_SEED,
    )
elif SPLIT_MODE == "tumor_blind":
    split_mode_note = "tumor_blind: this notebook implements cell-line-blind splits rather than Organ-level splits."
    split_unit_column = "cell_line"
    split_assignments = assign_group_blind_splits(
        examples_df,
        group_col="cell_line",
        split_fractions=SPLIT_FRACTIONS,
        seed=RANDOM_SEED,
    )
else:
    split_mode_note = "mixed: condition keys are held out, but every drug and cell line remains represented in train."
    split_unit_column = "condition_key"
    split_assignments = assign_mixed_split(
        examples_df,
        split_fractions=SPLIT_FRACTIONS,
        seed=RANDOM_SEED,
    )

split_examples_df = examples_df.copy()
split_examples_df["split"] = split_assignments.to_numpy()
validate_split_assignments(split_examples_df, SPLIT_MODE)

split_summary_df = build_split_summary(split_examples_df, SPLIT_FRACTIONS)
split_unit_summary_df = (
    split_examples_df.groupby("split")[split_unit_column]
    .nunique()
    .reindex(SPLIT_NAMES)
    .reset_index(name=f"unique_{split_unit_column}_count")
)
overlap_diagnostics_df = build_overlap_diagnostics(split_examples_df, SPLIT_MODE)

train_examples_df = split_examples_df.loc[split_examples_df["split"] == "train"].reset_index(drop=True)
val_examples_df = split_examples_df.loc[split_examples_df["split"] == "val"].reset_index(drop=True)
test_examples_df = split_examples_df.loc[split_examples_df["split"] == "test"].reset_index(drop=True)
split_sample_counts = {split_name: int((split_examples_df["split"] == split_name).sum()) for split_name in SPLIT_NAMES}

print(
    f"Using {SPLIT_MODE} split. {split_mode_note} Sample counts -> train: {split_sample_counts['train']}, val: {split_sample_counts['val']}, test: {split_sample_counts['test']}."
)
display(split_summary_df)
display(split_unit_summary_df)
display(overlap_diagnostics_df)


## Fit the Training Preprocessor

The `TrainingPreprocessor` computes three groups of statistics:

1. **`baseline_mean` / `baseline_std` / `normalized_dmso_input_expression_lookup`** over the full ~20k input gene space. Random forests are scale-invariant, so these are **not used** by the RF path. They are still computed to keep preprocessor output identical to notebook 4's Ridge/MLP runs so cross-model comparisons stay apples-to-apples.
2. **`dose_log_mean` / `dose_log_std`** for scaling the log10 concentration. The RF **does** use these via `build_rf_training_arrays`.
3. **`delta_mean` / `delta_std`** of the 977-dim target delta. The RF trains directly on raw delta (no standardization), so these are also unused here; they are kept for parity.

The fit is intentionally wasteful; the alternative (skipping it) would make RF results less directly comparable to notebook 4.


In [ ]:
training_preprocessor = TrainingPreprocessor(
    train_examples_df=train_examples_df,
    dmso_bundle=dmso_bundle,
    treatment_bundle=treatment_bundle,
    fingerprint_bundle=fingerprint_bundle,
    target_gene_indices=target_gene_indices,
    target_mode=TARGET_MODE,
    batch_size=PREPROCESS_BATCH_SIZE,
).fit()

preprocessor_summary_df = training_preprocessor.summary_frame()
display(preprocessor_summary_df)


## Assemble Random Forest Training Arrays

Build the flat `(X, y, baseline)` numpy arrays for each split using the preprocessor's cached lookup tensors. Optionally subsample training rows for smoke runs via `TRAIN_ROW_SUBSAMPLE_FRACTION`.


In [ ]:
X_train_full, y_train_full, baseline_train_full = build_rf_training_arrays(training_preprocessor, train_examples_df)
X_val, y_val, baseline_val = build_rf_training_arrays(training_preprocessor, val_examples_df)
X_test, y_test, baseline_test = build_rf_training_arrays(training_preprocessor, test_examples_df)

expected_input_dim = int(training_preprocessor.target_gene_dim + training_preprocessor.fingerprint_dim + 1)
expected_target_dim = int(training_preprocessor.target_gene_dim)

for split_name, X_split, y_split, baseline_split in (
    ("train", X_train_full, y_train_full, baseline_train_full),
    ("val", X_val, y_val, baseline_val),
    ("test", X_test, y_test, baseline_test),
):
    if X_split.shape[1] != expected_input_dim:
        raise ValueError(
            f"{split_name} X width {X_split.shape[1]} does not match expected input dim {expected_input_dim}."
        )
    if y_split.shape[1] != expected_target_dim:
        raise ValueError(
            f"{split_name} y width {y_split.shape[1]} does not match expected target gene dim {expected_target_dim}."
        )
    if baseline_split.shape != y_split.shape:
        raise ValueError(
            f"{split_name} baseline shape {baseline_split.shape} does not match y shape {y_split.shape}."
        )
    if np.any(baseline_split < 0):
        raise ValueError(f"{split_name} baseline contains negative values; expected log-normalized expression >= 0.")

if TRAIN_ROW_SUBSAMPLE_FRACTION is not None and TRAIN_ROW_SUBSAMPLE_FRACTION < 1.0:
    rng = np.random.default_rng(RANDOM_SEED)
    n_subsample = max(1, int(round(len(X_train_full) * float(TRAIN_ROW_SUBSAMPLE_FRACTION))))
    subsample_indices = rng.choice(len(X_train_full), size=n_subsample, replace=False)
    subsample_indices.sort()
    X_train = X_train_full[subsample_indices]
    y_train = y_train_full[subsample_indices]
    baseline_train = baseline_train_full[subsample_indices]
    train_examples_df_used = train_examples_df.iloc[subsample_indices].reset_index(drop=True)
else:
    X_train = X_train_full
    y_train = y_train_full
    baseline_train = baseline_train_full
    train_examples_df_used = train_examples_df

model_summary_df = pd.DataFrame(
    [
        {
            "model_family": "random_forest",
            "input_feature_dim": int(X_train.shape[1]),
            "target_gene_dim": int(y_train.shape[1]),
            "n_train_rows": int(X_train.shape[0]),
            "n_val_rows": int(X_val.shape[0]),
            "n_test_rows": int(X_test.shape[0]),
            "train_row_subsample_fraction": TRAIN_ROW_SUBSAMPLE_FRACTION,
            "rf_n_estimators": RF_N_ESTIMATORS,
            "rf_max_depth": RF_MAX_DEPTH,
            "rf_min_samples_leaf": RF_MIN_SAMPLES_LEAF,
            "rf_max_features": RF_MAX_FEATURES,
            "rf_n_jobs": RF_N_JOBS,
            "rf_oob_score": RF_OOB_SCORE,
            "treatment_feature_space": treatment_feature_space,
        }
    ]
)
print(
    f"Built RF arrays -> X_train {X_train.shape}, y_train {y_train.shape}, X_val {X_val.shape}, X_test {X_test.shape}."
)
display(model_summary_df)


## Train the Random Forest

A single call to `RandomForestRegressor.fit` does a multi-output regression over all 977 target genes simultaneously. Fit time scales roughly with `n_estimators * n_train_rows * log(n_train_rows) * sqrt(n_features) * n_outputs`; use `TRAIN_ROW_SUBSAMPLE_FRACTION` to throttle if needed.


In [ ]:
L.seed_everything(RANDOM_SEED, workers=True)

rf_kwargs = {
    "n_estimators": int(RF_N_ESTIMATORS),
    "max_depth": None if RF_MAX_DEPTH is None else int(RF_MAX_DEPTH),
    "min_samples_leaf": int(RF_MIN_SAMPLES_LEAF),
    "max_features": RF_MAX_FEATURES,
    "n_jobs": int(RF_N_JOBS),
    "oob_score": bool(RF_OOB_SCORE),
    "random_state": int(RANDOM_SEED),
}
rf_estimator = RandomForestRegressor(**rf_kwargs)

fit_start = time.perf_counter()
rf_estimator.fit(X_train, y_train)
fit_seconds = float(time.perf_counter() - fit_start)

run_name = f"{SPLIT_MODE}_{MODEL_NAME}_{TARGET_MODE}"
sklearn_output_dir = build_project_path(SKLEARN_OUTPUT_DIR)
run_output_dir = sklearn_output_dir / run_name
run_output_dir.mkdir(parents=True, exist_ok=True)

estimator_path = run_output_dir / "random_forest.joblib"
joblib.dump(rf_estimator, estimator_path)

oob_score = float(rf_estimator.oob_score_) if RF_OOB_SCORE else None
training_run_summary = {
    "run_name": run_name,
    "estimator_path": str(estimator_path),
    "fit_seconds": fit_seconds,
    "oob_score": oob_score,
    "n_train_rows": int(X_train.shape[0]),
    "n_val_rows": int(X_val.shape[0]),
    "n_test_rows": int(X_test.shape[0]),
    "rf_kwargs": rf_kwargs,
    "train_row_subsample_fraction": TRAIN_ROW_SUBSAMPLE_FRACTION,
    "split_mode": SPLIT_MODE,
    "target_mode": TARGET_MODE,
}
summary_path = run_output_dir / "training_run_summary.json"
summary_path.write_text(json.dumps(training_run_summary, indent=2, default=str))

training_run_summary_df = pd.DataFrame([training_run_summary])
display(training_run_summary_df)


## Evaluate the Fitted Random Forest

Uses `evaluate_sklearn_predictions`, which produces the same `(metrics, inspection_df, prediction_details_df)` schema as `evaluate_model_on_loader` in notebook 4.


In [ ]:
predicted_delta_by_split = {
    "train": rf_estimator.predict(X_train),
    "val": rf_estimator.predict(X_val),
    "test": rf_estimator.predict(X_test),
}
split_arrays = {
    "train": (X_train, y_train, baseline_train, train_examples_df_used),
    "val": (X_val, y_val, baseline_val, val_examples_df),
    "test": (X_test, y_test, baseline_test, test_examples_df),
}

evaluation_rows = []
inspection_tables = {}
prediction_detail_tables = {}
for split_name, (_, y_split, baseline_split, metadata_split) in split_arrays.items():
    metrics_row, inspection_df, prediction_details_df = evaluate_sklearn_predictions(
        predicted_delta_np=predicted_delta_by_split[split_name],
        target_delta_np=y_split,
        baseline_np=baseline_split,
        metadata_df=metadata_split,
        split_name=split_name,
        gene_ids=target_gene_ids,
    )
    evaluation_rows.append(metrics_row)
    inspection_tables[split_name] = inspection_df
    prediction_detail_tables[split_name] = prediction_details_df

evaluation_summary_df = pd.DataFrame(evaluation_rows)
test_prediction_details_df = prediction_detail_tables["test"].copy()
print(f"Evaluated fitted random forest from {estimator_path}")
display(evaluation_summary_df)
display(inspection_tables["test"])
display(test_prediction_details_df.head())


## Training Summary and Train/Val Diagnostic Tables

Random Forest has no per-epoch loss, so the train/val line plot from notebook 4 is replaced by a training-run summary plus the same Mann-Whitney and DEG diagnostic tables, and a bar plot of mean Mann-Whitney p-value.


In [ ]:
train_val_mann_whitney_summary_df = (
    evaluation_summary_df.loc[evaluation_summary_df["split"].isin(["train", "val"])]
    .copy()
    .sort_values("split", key=lambda s: s.map({"train": 0, "val": 1}), ignore_index=True)
)

display(training_run_summary_df)
display(
    train_val_mann_whitney_summary_df[
        [
            "split",
            "mann_whitney_pvalue_mean",
            "mann_whitney_pvalue_median",
            "mann_whitney_not_significant_fraction",
        ]
    ]
)
display(
    train_val_mann_whitney_summary_df[
        [
            "split",
            "top50_deg_match_count_mean",
            "top50_deg_match_count_median",
            "top50_deg_match_fraction_mean",
            "top50_deg_match_fraction_median",
            "signed_ndcg_at_50_mean",
            "signed_ndcg_at_50_median",
        ]
    ]
)

fig, pvalue_ax = plt.subplots(figsize=(8, 6))
sns.barplot(
    data=train_val_mann_whitney_summary_df,
    x="split",
    y="mann_whitney_pvalue_mean",
    hue="split",
    order=["train", "val"],
    hue_order=["train", "val"],
    palette={"train": "#1f77b4", "val": "#d62728"},
    dodge=False,
    legend=False,
    ax=pvalue_ax,
)
pvalue_ax.axhline(0.05, color="#444444", linestyle="--", linewidth=1.2, label="p = 0.05")
pvalue_ax.set_title(f"Random Forest Mean Mann-Whitney P-Value ({run_name})")
pvalue_ax.set_xlabel("Split")
pvalue_ax.set_ylabel("Average p-value")
pvalue_ax.set_ylim(0, max(1.0, float(train_val_mann_whitney_summary_df["mann_whitney_pvalue_mean"].max()) * 1.1))
pvalue_ax.grid(True, axis="y", alpha=0.25)
pvalue_ax.legend(frameon=False, loc="upper right")

fig.tight_layout()
plt.show()


## Plot Predicted vs Actual Test Pairs


In [ ]:
sampled_test_prediction_details_df = sample_prediction_details(
    prediction_details_df=test_prediction_details_df,
    sample_count=PLOT_TEST_SAMPLE_COUNT,
    sample_strategy=PLOT_TEST_SAMPLE_STRATEGY,
    random_seed=PLOT_RANDOM_SEED,
)
test_prediction_plot_df, sampled_test_pair_summary_df, test_embedding_explained_variance_ratio = (
    build_sklearn_prediction_pair_embedding(
        sampled_prediction_details_df=sampled_test_prediction_details_df,
        baseline_np=baseline_test,
        predicted_delta_np=predicted_delta_by_split["test"],
        target_delta_np=y_test,
        embedding_method=EMBEDDING_METHOD,
        n_components=N_EMBEDDING_COMPONENTS,
    )
)

print(
    f"Plotted {len(sampled_test_prediction_details_df)} {PLOT_TEST_SAMPLE_STRATEGY} test pairs with {EMBEDDING_METHOD.upper()} using the fitted random forest."
)
display(sampled_test_pair_summary_df)

fig, ax = plt.subplots(figsize=(12, 9))
for pair_index, pair_points_df in test_prediction_plot_df.groupby("pair_index", sort=True):
    ordered_pair_points_df = pair_points_df.set_index("point_kind").loc[["actual", "predicted"]]
    ax.plot(
        ordered_pair_points_df["embedding_1"],
        ordered_pair_points_df["embedding_2"],
        color="#9e9e9e",
        linewidth=0.8,
        alpha=0.65,
        zorder=1,
    )

sns.scatterplot(
    data=test_prediction_plot_df,
    x="embedding_1",
    y="embedding_2",
    hue="point_kind",
    style="point_kind",
    palette={"actual": "#1f77b4", "predicted": "#d62728"},
    markers={"actual": "o", "predicted": "X"},
    s=90,
    alpha=0.9,
    ax=ax,
)
ax.set_title(
    f"Predicted vs Actual Test Expression in {EMBEDDING_METHOD.upper()} Space ({len(sampled_test_prediction_details_df)} pairs, {run_name})"
)
ax.set_xlabel(
    f"{EMBEDDING_METHOD.upper()} 1 ({test_embedding_explained_variance_ratio[0] * 100:.1f}% var)"
)
ax.set_ylabel(
    f"{EMBEDDING_METHOD.upper()} 2 ({test_embedding_explained_variance_ratio[1] * 100:.1f}% var)"
)
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout()
plt.show()


## Plot Test Cell-Line Accuracy


In [ ]:
test_prediction_details_df = test_prediction_details_df.copy()
test_prediction_details_df["is_correct_prediction"] = test_prediction_details_df["mann_whitney_pvalue"] > 0.05

test_cell_line_accuracy_df = (
    test_prediction_details_df.groupby("cell_line", as_index=False)
    .agg(
        n_test_samples=("condition_key", "size"),
        n_correct_predictions=("is_correct_prediction", "sum"),
    )
)
test_cell_line_accuracy_df["n_correct_predictions"] = test_cell_line_accuracy_df["n_correct_predictions"].astype(int)
test_cell_line_accuracy_df["cell_line_accuracy"] = (
    test_cell_line_accuracy_df["n_correct_predictions"] / test_cell_line_accuracy_df["n_test_samples"]
)
test_cell_line_accuracy_df = test_cell_line_accuracy_df.sort_values(
    ["cell_line_accuracy", "cell_line"],
    ascending=[False, True],
    ignore_index=True,
)
mean_cell_line_accuracy = float(test_cell_line_accuracy_df["cell_line_accuracy"].mean())

print("Computed test-set cell-line accuracy using mann_whitney_pvalue > 0.05 as the correctness rule.")
display(test_cell_line_accuracy_df)

fig, ax = plt.subplots(figsize=(14, 6))
sns.barplot(
    data=test_cell_line_accuracy_df,
    x="cell_line",
    y="cell_line_accuracy",
    color="#4c72b0",
    ax=ax,
)
ax.axhline(
    mean_cell_line_accuracy,
    color="#d62728",
    linestyle="--",
    linewidth=1.5,
    label=f"Mean accuracy = {mean_cell_line_accuracy:.3f}",
)
ax.set_title("Test Cell-Line Accuracy from Mann-Whitney Non-Significance")
ax.set_xlabel("Cell line")
ax.set_ylabel("Accuracy")
ax.set_ylim(0, 1)
ax.grid(True, axis="y", alpha=0.25)
ax.tick_params(axis="x", rotation=45)
ax.legend(frameon=False, loc="upper right")
fig.tight_layout()
plt.show()


## Plot Test Cell-Line DEG Overlap Metrics


In [ ]:
test_cell_line_deg_match_df = (
    test_prediction_details_df.groupby("cell_line", as_index=False)
    .agg(
        n_test_samples=("condition_key", "size"),
        mean_top50_deg_match_count=("top50_deg_match_count", "mean"),
        median_top50_deg_match_count=("top50_deg_match_count", "median"),
        min_top50_deg_match_count=("top50_deg_match_count", "min"),
        max_top50_deg_match_count=("top50_deg_match_count", "max"),
    )
    .sort_values(["median_top50_deg_match_count", "cell_line"], ascending=[False, True], ignore_index=True)
)
test_cell_line_signed_ndcg_df = (
    test_prediction_details_df.groupby("cell_line", as_index=False)
    .agg(
        n_test_samples=("condition_key", "size"),
        mean_signed_ndcg_at_50=("signed_ndcg_at_50", "mean"),
        median_signed_ndcg_at_50=("signed_ndcg_at_50", "median"),
        min_signed_ndcg_at_50=("signed_ndcg_at_50", "min"),
        max_signed_ndcg_at_50=("signed_ndcg_at_50", "max"),
    )
    .sort_values(["median_signed_ndcg_at_50", "cell_line"], ascending=[False, True], ignore_index=True)
)
deg_match_order = test_cell_line_deg_match_df["cell_line"].tolist()
signed_ndcg_order = test_cell_line_signed_ndcg_df["cell_line"].tolist()

print("Computed test-set top-50 DEG overlap counts and signed nDCG@50 from predicted vs true delta expression.")
display(test_cell_line_deg_match_df)
display(test_cell_line_signed_ndcg_df)

fig, axes = plt.subplots(1, 2, figsize=(20, 7))
deg_ax, ndcg_ax = axes

sns.boxplot(
    data=test_prediction_details_df,
    x="cell_line",
    y="top50_deg_match_count",
    order=deg_match_order,
    whis=(0, 100),
    color="#4c72b0",
    ax=deg_ax,
)
deg_ax.set_title("Test Cell-Line Top-50 DEG Overlap Counts")
deg_ax.set_xlabel("Cell line")
deg_ax.set_ylabel("Matched DEGs out of 50")
deg_ax.grid(True, axis="y", alpha=0.25)
deg_ax.tick_params(axis="x", rotation=45)

sns.boxplot(
    data=test_prediction_details_df,
    x="cell_line",
    y="signed_ndcg_at_50",
    order=signed_ndcg_order,
    whis=(0, 100),
    color="#55a868",
    ax=ndcg_ax,
)
ndcg_ax.set_title("Test Cell-Line Signed nDCG@50")
ndcg_ax.set_xlabel("Cell line")
ndcg_ax.set_ylabel("Signed nDCG@50")
ndcg_ax.set_ylim(0, 1)
ndcg_ax.grid(True, axis="y", alpha=0.25)
ndcg_ax.tick_params(axis="x", rotation=45)

fig.tight_layout()
plt.show()
